# Agentic Rag Module

- Note: This a local AI setup using LM Studio and a downloaded model for LLM generation.

In [1]:
# Importing libraries and create openai client
import os
from dotenv import load_dotenv
load_dotenv()
from openai import OpenAI
import requests


In [2]:
openai_client = OpenAI(
    api_key=os.getenv("LMSTUDIO_API_KEY"),
    base_url=os.getenv("LMSTUDIO_HOST")
)

In [3]:
# Defining our function to talk the LLM
def llm(prompt):
    response = openai_client.chat.completions.create(
        model="qwen/qwen3.5-9b",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

In [4]:
# Testing the LLM
llm("What's the answer to life?")

'The "answer to life" is a famous question that has been explored from many different angles, depending on whether you look for a scientific fact, a philosophical insight, or even a pop-culture reference.\n\nHere are the most common answers:\n\n**1. The Pop-Culture Answer**\nIf you are thinking of Douglas Adams\' *The Hitchhiker\'s Guide to the Galaxy*, the answer is **42**.\nIn the story, a supercomputer named Deep Thought calculates it for 7.5 million years. The twist in the book is that while the *Answer* was known, the actual *Question* ("What is life all about?") was never asked before calculation began.\n\n**2. The Philosophical & Scientific Answer**\nThere is no single factual number or word that applies to every person. Instead, most thinkers suggest that the answer is subjective and found within yourself:\n*   **Existentialism**: Life has no inherent meaning; you are free to create your own purpose through your choices and actions.\n*   **Humanism**: The meaning of life is fou

In [5]:
# Demo how LLM provides an answer with little context in mind
question = "I just discovered the course. Can I join now?"
answer = llm(question)
print(answer)

Congratulations on discovering a new course! Whether you can join **right now** depends entirely on the specific platform or institution offering it.

Here are a few things to consider:

*   **Is enrollment open?** Some courses run on fixed schedules with specific start dates. If the current term is full or closed, you might need to wait for the next cohort or look for a "waitlist" option.
*   **Self-Paced vs. Live:** Many modern courses are self-paced (available 24/7), while others require you to join live sessions at specific times. Check if there are any registration deadlines.
*   **Free Trial or Audit Mode:** Some platforms (like Coursera or edX) allow you to start a course immediately for free (as an "auditor") even if the full certificate version is behind a paywall or closed enrollment.

**What would be most helpful right now?**
If you tell me the **name of the course** or the **platform/website** where you found it, I can help you check their typical enrollment policies or sug

In [6]:
# Adding context
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [7]:
# Building a prompt that includes question and context:
prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [8]:
# Updated answer with context; this is Retrival Augmented Generation (RAG)
answer = llm(prompt)
print(answer)

Yes, you can join the course now. However, please note that if you want to receive a certificate, you need to submit your project while submissions are still being accepted.


In [9]:
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

Diagram showing how RAG works:

![RAG Diagram](./docs/images/rag_diagram.png)

In [10]:
# Importing external data through a strucuture format for LLM to use
docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [11]:
display(courses_raw)

[{'course': 'data-engineering-zoomcamp',
  'course_name': 'Data Engineering Zoomcamp',
  'path': '/json/data-engineering-zoomcamp.json',
  'questions_count': 402},
 {'course': 'stock-markets-analytics-zoomcamp',
  'course_name': 'Stock Markets Analytics Zoomcamp',
  'path': '/json/stock-markets-analytics-zoomcamp.json',
  'questions_count': 93},
 {'course': 'ai-dev-tools-zoomcamp',
  'course_name': 'AI Dev Tools Zoomcamp',
  'path': '/json/ai-dev-tools-zoomcamp.json',
  'questions_count': 41},
 {'course': 'llm-zoomcamp',
  'course_name': 'LLM Zoomcamp',
  'path': '/json/llm-zoomcamp.json',
  'questions_count': 79},
 {'course': 'mlops-zoomcamp',
  'course_name': 'MLOps Zoomcamp',
  'path': '/json/mlops-zoomcamp.json',
  'questions_count': 255},
 {'course': 'machine-learning-zoomcamp',
  'course_name': 'ML Zoomcamp',
  'path': '/json/machine-learning-zoomcamp.json',
  'questions_count': 472}]

In [12]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1342

Each entry has:

    id - unique identifier
    course - course slug (e.g., machine-learning-zoomcamp)
    section - which section of the course
    question - the FAQ question
    answer - the FAQ answer


In [13]:
documents[0]

{'id': '9e508f2212',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: When does the course start?',
 'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."}

### Building a Search Function in the RAG Pipeline

In [14]:
# Building the Search Index
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [15]:
# Demo-ing search with a question
question = "I just discovered the course. Can I join now?"

search_results = index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5}, # assigning weights to specific fields
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [16]:
# Reviewing all questions in faq search_results
[doc["question"] for doc in search_results]

['I just discovered the course. Can I still join?',
 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
 'When will the course be offered next?',
 'I missed the first homework - can I still get a certificate?']

In [17]:
# Filtering results by course
results = index.search(
    question,
    num_results=5,
    filter_dict={"course": "mlops-zoomcamp"}
)

In [18]:
results

[{'id': '2d8b16c2a0',
  'course': 'mlops-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course - Can I still join the course after the start date?',
  'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute."},
 {'id': 'c842475338',
  'course': 'mlops-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Homework: Just found this course, can I still submit homeworks?',
  'answer': 'To clarify on **late homework submissions**:\n\n- You cannot submit after the homework is scored, as the form is closed.\n- Once the form is closed (i.e., scored), no further submissions are possible.\n- You can check your code against the solution by reviewing the `homework.md` file.\n\nIf the due date has passed but the form is still "O

In [19]:
# Return all questions listed in search results and related to user initial query
[doc["question"] for doc in results]


['Course - Can I still join the course after the start date?',
 'Homework: Just found this course, can I still submit homeworks?',
 'I forgot if I registered, can I still join the zoomcamp?',
 'Certificate - Can I follow the course in a self-paced mode and get a certificate?',
 'Course: How do I start?']

In [20]:
# Create a executable function
def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

In [21]:
# Now we can call the search function as part of RAG pipeline
search_results = search(question)

In [22]:
search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

### Building the Prompt

In [23]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

In [24]:
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [ ]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [27]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [28]:
prompt = build_prompt(question, search_results)

print(prompt)

Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project

The prompt is the bridge between search and the LLM. A bad prompt lets the LLM ignore the context and hallucinate. A good prompt keeps the answer grounded.

### The LLM

In [29]:
# Sending the prompt to the LLM
response = openai_client.chat.completions.create(
        model="qwen/qwen3.5-9b",
        messages=[{"role": "user", "content": prompt}]
    )

In [33]:
response

ChatCompletion(id='chatcmpl-s15wgjrq29ab3pb5y7s97', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Yes, you can join the course now! However, please note that if your goal is to receive an official **certificate**, you must submit your final project while the submission window is still open. Additionally, certificates are only awarded for courses run with a "live" cohort (not in self-paced mode), so joining now ensures you can participate in the required peer-review process along with your peers.', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[], reasoning_content=''))], created=1781309301, model='qwen/qwen3.5-9b', object='chat.completion', moderation=None, service_tier=None, system_fingerprint='qwen/qwen3.5-9b', usage=CompletionUsage(completion_tokens=80, prompt_tokens=359, total_tokens=439, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_token

In [31]:
response.choices[0]

Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Yes, you can join the course now! However, please note that if your goal is to receive an official **certificate**, you must submit your final project while the submission window is still open. Additionally, certificates are only awarded for courses run with a "live" cohort (not in self-paced mode), so joining now ensures you can participate in the required peer-review process along with your peers.', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[], reasoning_content=''))

In [35]:
response.choices[0].message.content

'Yes, you can join the course now! However, please note that if your goal is to receive an official **certificate**, you must submit your final project while the submission window is still open. Additionally, certificates are only awarded for courses run with a "live" cohort (not in self-paced mode), so joining now ensures you can participate in the required peer-review process along with your peers.'

In [36]:
response.usage

CompletionUsage(completion_tokens=80, prompt_tokens=359, total_tokens=439, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=0, rejected_prediction_tokens=None), prompt_tokens_details=None)

In [ ]:
# Calcuating OpenAI cost
input_price = 0.75 / 1_000_000
output_price = 4.50 / 1_000_000

cost = (
    response.usage.prompt_tokens * input_price +
    response.usage.completion_tokens * output_price
)

cost

0.00062925

Previously we sent only one string as input and got back a response. In practice, you typically send a message history - a list of messages where each message has a role.

Think of a ChatGPT conversation. It starts with a hidden system prompt that tells the LLM how to behave, one you never see. After that, your messages and the LLM's replies alternate. The LLM has no memory of its own, so it needs the full history passed in to continue the conversation.

We send two messages:

    developer - system-level instructions (how the LLM should behave)
    user - the actual prompt with the question and context


In [38]:
message_history = [
    {"role": "developer", "content": INSTRUCTIONS},
    {"role": "user", "content": prompt}
]

response = openai_client.chat.completions.create(
        model="qwen/qwen3.5-9b",
        messages=message_history
    )

This separates the fixed instructions from the user prompt, which changes every request.

OpenAI accepts both developer and system for the instruction role. There may be some difference between them, but in practice I don't see it change the result either way.

In [39]:
# Building the LLM function
def llm(instructions, user_prompt, model="qwen/qwen3.5-9b"):
    message_history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.chat.completions.create(
        model=model,
        messages=message_history
    )
    
    return response.choices[0].message.content

In [40]:
# Full RAG
def rag(query, model="qwen/qwen3.5-9b"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

In [41]:
answer = rag("I just discovered the course. Can I join now?")
print(answer)

Yes, you can join the course now. However, please note that if you wish to receive a certificate, you must submit your project while the submissions are still being accepted. Additionally, certificates are only available for those who finish the course with a "live" cohort (not in self-paced mode).


In [42]:
rag("How do I get a certificate?")

'To get a certificate, you must finish the course with a **"live" cohort**. You cannot receive a certificate if you complete the course in self-paced mode because certificates require peer-reviewing three capstone projects, which can only be done during the live course when peer review lists are compiled. Additionally, while homework is recommended, it is not mandatory; you simply need to pass the Capstone project to receive your certificate.'

In [44]:
rag("how do I submit homework assignments")

'Based on the context provided, here is how you can submit homework assignments:\n\n*   **Choose a login provider:** Due to sporadic errors with GitHub logging, it is recommended to use **Google** or **Slack** to log in and submit your homework answers.\n*   **Follow the submission windows:** There are specific "attempt#1" and "attempt#2" submission windows. If you miss the first window, you can catch up during the second.\n    *   *Note:* For the Capstone Project specifically, Attempt 1 serves as your initial project. If that submission fails (or if you want to submit a second project for extra exposure), you must use **different datasets and problem statements** for the second attempt.\n*   **Participate in Peer Review:** Your submission will not count towards your certification unless you participate in the peer-review of three peers in your cohort.'

Notice how the answers reference specific courses and sections. The LLM reads from our knowledge base before answering - that's how RAG works.
This approach is modular. You can swap the search backend, the prompt template, or the LLM model. Nothing else needs to change. Later when we replace minsearch with sqlitesearch, only the search function changes.